In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS olist")
spark.sql("USE CATALOG olist")
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
spark.sql("USE DATABASE bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS olist.default.landing")

In [0]:
LANDING_PATH = "/Volumes/olist/default/landing"

In [0]:
from pyspark.sql import functions as F

def ingest_csv(file_name: str, table_name: str) -> None:
    """
    Lê um arquivo CSV da Landing Zone e salva como Delta Table na camada Bronze,
    adicionando a coluna de timestamp de ingestão conforme exigido pela atividade.
    
    Args:
        file_name: Nome do arquivo CSV (ex: 'olist_customers_dataset.csv')
        table_name: Nome da tabela de destino na Bronze (ex: 'tb_customers')
    """
    file_path = f"{LANDING_PATH}/{file_name}"
    
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )
    
    df = df.withColumn("timestamp_ingestion", F.current_timestamp())
    
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"olist.bronze.{table_name}")
    )
    
    print(f"✅ {file_name} -> olist.bronze.{table_name} ({df.count()} registros)")

In [0]:
# Mapeamento de arquivos para tabelas 
arquivos = [
    ("olist_customers_dataset.csv",            "tb_customers"),
    ("olist_geolocation_dataset.csv",          "tb_geolocalizacao"),
    ("olist_order_items_dataset.csv",          "tb_order_items"),
    ("olist_order_payments_dataset.csv",       "tb_order_payments"),
    ("olist_order_reviews_dataset.csv",        "tb_order_reviews"),
    ("olist_orders_dataset.csv",               "tb_orders"),
    ("olist_products_dataset.csv",             "tb_products"),
    ("olist_sellers_dataset.csv",              "tb_sellers"),
    ("product_category_name_translation.csv",  "tb_product_category_name_translation"),
]

for file_name, table_name in arquivos:
    ingest_csv(file_name, table_name)